In [10]:
# =========================================================
# COSINE SIMILARITY
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity

from src.preprocessing.clean_text import clean_text
from src.preprocessing.casefolding import casefolding
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

from supabase import create_client, Client
import uuid
from datetime import datetime


In [11]:
# =========================================================
# KONEKSI SUPABASE
# =========================================================
SUPABASE_URL = 'https://bnuzmrtiaciqlotxcgot.supabase.co'
SUPABASE_KEY = 'sb_publishable_Z8M8GISPVKMp1SGrQHrlLg_AZa8EOo-'
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Koneksi Supabase berhasil!')


✅ Koneksi Supabase berhasil!


In [12]:
# =========================================================
# LOAD DATA
# =========================================================
base_dir = os.path.abspath('..')
tfidf_dir = os.path.join(base_dir, 'data', 'tfidf')

# Load vectorizer
with open(os.path.join(tfidf_dir, 'vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)

# Load matriks TF-IDF
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, 'tfidf_matrix.npz'))

# Load data artikel lengkap
doc_index = pd.read_csv(os.path.join(base_dir, 'data', 'cleaned_papers.csv'))

stop_words = get_stopwords()

print(f'✅ Matriks TF-IDF: {tfidf_matrix.shape}')
print(f'✅ Dokumen: {len(doc_index)} artikel')


✅ Matriks TF-IDF: (200, 1722)
✅ Dokumen: 200 artikel


d:\Tugas Akhir\paperCi\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Tugas Akhir\paperCi\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
# =========================================================
# PREPROCESSING QUERY
# =========================================================
def preprocess_query(query: str) -> str:
    text = clean_text(query)
    text = casefolding(text)
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    # Skip stemming kalau query berbahasa Inggris
    if all(token.isascii() for token in tokens):
        return ' '.join(tokens)
    else:
        tokens = stemming(tokens)  # hanya untuk Indonesia
        return ' '.join(tokens)

In [14]:
def is_direct_pdf(url: str) -> bool:
    if pd.isna(url):
        return False

    url = url.lower()

    return (
        url.endswith(".pdf")
        or "/pdf/" in url
        or "arxiv.org/pdf" in url
        or "pmc.ncbi.nlm.nih.gov" in url
        or "jmlr.org" in url
    )

In [15]:
# =========================================================
# FUNGSI PENCARIAN
# =========================================================
def count_term_frequency(text: str, query_terms: list) -> int:
    """Hitung jumlah kemunculan query terms dalam teks dokumen"""
    
    text_tokens = str(text).lower().split()
    
    return sum(
        text_tokens.count(term)
        for term in query_terms
    )


def search(query: str, top_k: int = 10):

    processed_query = preprocess_query(query)
    query_terms = processed_query.split()

    query_vector = vectorizer.transform([processed_query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = doc_index.iloc[ranked_indices][
        [
            'id','title','abstract','authors','year',
            'source','category','pdf_url','url','scrape_status'
        ]
    ].copy()

    results['similarity_score'] = scores[ranked_indices]
    results['abstract'] = results['abstract'].fillna("")

    # FIX PDF
    results['is_pdf'] = results['pdf_url'].apply(is_direct_pdf)

    results['access_url'] = results.apply(
        lambda r: r['pdf_url'] if r['is_pdf'] else r['url'],
        axis=1
    )

    results = results.reset_index(drop=True)
    results['rank'] = range(1, len(results) + 1)

    results['term_frequency'] = results['abstract'].apply(
        lambda x: count_term_frequency(x, query_terms)
    )

    return results

print('✅ Fungsi search siap!')

✅ Fungsi search siap!


In [16]:
def save_statistics_to_supabase(results: pd.DataFrame, query: str):
    if results.empty:
        print(f'⚠️ Tidak ada data untuk query "{query}"')
        return None

    total_occurrences = int(results['term_frequency'].sum())
    paper_count = int((results['term_frequency'] > 0).sum())

    record = {
        'id': str(uuid.uuid4()),
        'query': query,
        'total_occurrences': total_occurrences,
        'paper_count': paper_count,
        'created_at': datetime.utcnow().isoformat()
    }

    try:
        response = supabase.table('query_statistics').insert(record).execute()
        print(response)  # ✅ cek hasil insert
        print(f'✅ Statistik tersimpan untuk query: "{query}"')
        return record['id']
    except Exception as e:
        print(f'❌ Gagal simpan statistik: {e}')
        return None


In [17]:
def save_results_to_supabase(results: pd.DataFrame, query_id: str, query: str):
    if results.empty:
        print(f'⚠️ Tidak ada data untuk disimpan ke similarity_results')
        return

    records = []
    for _, row in results.iterrows():
        records.append({
            'id'              : str(uuid.uuid4()),
            'article_id'      : int(row['id']),
            'query_id'        : query_id,   # ✅ relasi ke query_statistics
            'compared_text'   : query,
            'similarity_score': float(round(row['similarity_score'], 6)),
            'created_at'      : datetime.utcnow().isoformat()
        })

    try:
        response = supabase.table('similarity_results').upsert(records).execute()
        print(f'✅ {len(records)} data tersimpan ke similarity_results (query: "{query}")')
    except Exception as e:
        print(f'❌ Gagal simpan ke Supabase: {e}')


In [18]:
# =========================================================
# JALANKAN SEARCH + SIMPAN CSV + SIMPAN SUPABASE
# =========================================================
queries_proposal = [
    "machine learning",
    "web development",
    "cyber security",
    "mobile application"
]

results_dir = os.path.join(base_dir, 'data', 'cosine_results')
os.makedirs(results_dir, exist_ok=True)

for query in queries_proposal:
    print(f'\n{"="*60}')
    results = search(query, top_k=10)

    if results.empty:
        print(f'⚠️ Tidak ada hasil untuk query: "{query}"')
        filename  = f'similarity_score_{query.replace(" ", "_")}.csv'
        save_path = os.path.join(results_dir, filename)
        results.to_csv(save_path, index=False)
        print(f'📂 File kosong tetap dibuat: {save_path}')
        continue

    print(f'\n{"Rank":<5} {"Score":>8} {"TF":>5}  Judul')
    print('-'*80)
    for _, row in results.iterrows():
        print(f'{int(row["rank"]):<5} {row["similarity_score"]:>8.4f} {row["term_frequency"]:>5}  {str(row["title"])[:55]}')

    # Simpan ke CSV
    filename  = f'similarity_score_{query.replace(" ", "_")}.csv'
    save_path = os.path.join(results_dir, filename)
    results.to_csv(save_path, index=False)
    print(f'\n✅ CSV tersimpan : {save_path}')
    print(f'📊 Total dokumen relevan: {len(results)}')

    # Simpan summary ke query_statistics
    query_id = save_statistics_to_supabase(results, query)

    # Simpan detail ke similarity_results
    if query_id:
        save_results_to_supabase(results, query_id, query)

print('\n🎉 Semua selesai!')




Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.6232     7  Machine learning and deep learning: A review of methods
2       0.6099     6  When machine learning meets privacy: A survey and outlo
3       0.5976     9  Machine learning and deep learning: C. Janiesch et al.
4       0.5638     7  An overview of machine learning classification techniqu
5       0.5608     5  Machine learning foundations
6       0.5598     6  Scientific machine learning benchmarks
7       0.5578     7  Financial applications of machine learning: A literatur
8       0.5326     4  Machine learning in chemical engineering: A perspective
9       0.5235     6  Machine learning and applications in microbiology
10      0.5064     6  Using machine learning to detect misstatements

✅ CSV tersimpan : d:\Tugas Akhir\paperci_artikel\backend\data\cosine_results\similarity_score_machine_learning.csv
📊 Total dokumen relevan: 10


C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\2422647348.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()


data=[{'id': '93671cee-0632-48d1-9134-f49e15f13bb8', 'query': 'machine learning', 'total_occurrences': 63, 'paper_count': 10, 'created_at': '2026-05-20T20:56:56.41268+00:00'}] count=None
✅ Statistik tersimpan untuk query: "machine learning"
✅ 10 data tersimpan ke similarity_results (query: "machine learning")


Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.5223     3  Accessible web development: Opportunities to improve th
2       0.4932     4  AI AND WEB DEVELOPMENT
3       0.4866     3  The rise of disappearing frameworks in web development
4       0.4283     4  Llms in web development: Evaluating llm-generated php c
5       0.4213     4  Machine learning for web development: A fusion
6       0.4188     4  Cognitive disabilities and web accessibility: a survey 
7       0.4046     2  Web Developer and Tech Preneurs
8       0.4032     4  Study on mvc framework for web development in php
9       0.3991     5  Progr

C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\545450157.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at'      : datetime.utcnow().isoformat()
C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\2422647348.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()


data=[{'id': 'b54c03e1-b95c-41b0-b270-92f440f2aeab', 'query': 'web development', 'total_occurrences': 37, 'paper_count': 10, 'created_at': '2026-05-20T20:56:57.537079+00:00'}] count=None
✅ Statistik tersimpan untuk query: "web development"
✅ 10 data tersimpan ke similarity_results (query: "web development")


Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.8096    12  A systematic literature review on the cyber security
2       0.6593     8  Analysis of cyber security knowledge gaps based on cybe
3       0.5951     5  The difference between cyber security vs information se
4       0.5903     8  A comprehensive review of cyber security vulnerabilitie
5       0.5828     6  The role of cyber security in a digitalizing economy: a
6       0.5756     8  Defining cyber security and cyber security risk within 
7       0.5468     5  Evolution of the cyber security threat: an overview of 
8       0.5115     6  Cyber resilienc

...

data=[{'id': '170dc672-5309-477c-969c-ef5569b274b2', 'query': 'cyber security', 'total_occurrences': 67, 'paper_count': 10, 'created_at': '2026-05-20T20:56:57.812946+00:00'}] count=None
✅ Statistik tersimpan untuk query: "cyber security"
✅ 10 data tersimpan ke similarity_results (query: "cyber security")



C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\545450157.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at'      : datetime.utcnow().isoformat()



Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.4751     4  Deep learning methods for accurate skin cancer recognit
2       0.4286     6  User interface design & evaluation of mobile applicatio
3       0.4285     4  User experience analysis on mobile application design u
4       0.4227     5  Consumer adoption of the Uber mobile application: Insig
5       0.4128     4  A new mobile application of agricultural pests recognit
6       0.3937     4  Design and implementation of smart hydroponics farming 
7       0.3795     4  Towards a new learning experience through a mobile appl
8       0.3526     4  Human monkeypox classification from skin lesion images 
9       0.3175     4  The role of mobile application acceptance in shaping e-
10      0.3146     4  Enhancing English vocabulary learning through mobile ap

✅ CSV tersimpan : d:\Tugas Akhir\paperci_artikel\backend\data\cosine_results\similarity_score_mobile_applicat

C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\2422647348.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()
C:\Users\hp_\AppData\Local\Temp\ipykernel_15772\545450157.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at'      : datetime.utcnow().isoformat()


✅ 10 data tersimpan ke similarity_results (query: "mobile application")

🎉 Semua selesai!
